# Instance Segmentation — Mask R-CNN Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: RoIAlign from scratch

This is the one component of Mask R-CNN that is simpler to understand as code than as prose.

In [ ]:
```python

import torch

import torch.nn.functional as F

def roi_align_single(feature, box, output_size=7, spatial_scale=1 / 16.0):

    """

    feature: (C, H, W) single-image feature map

    box: (x1, y1, x2, y2) in original image pixel coordinates

    output_size: side of the output grid (7 for box head, 14 for mask head)

    spatial_scale: reciprocal of the feature map stride

    """

    C, H, W = feature.shape

    x1, y1, x2, y2 = [c * spatial_scale - 0.5 for c in box]

    bin_w = (x2 - x1) / output_size

    bin_h = (y2 - y1) / output_size

    grid_y = torch.linspace(y1 + bin_h / 2, y2 - bin_h / 2, output_size)

    grid_x = torch.linspace(x1 + bin_w / 2, x2 - bin_w / 2, output_size)

    yy, xx = torch.meshgrid(grid_y, grid_x, indexing="ij")

    gx = 2 * (xx + 0.5) / W - 1

    gy = 2 * (yy + 0.5) / H - 1

    grid = torch.stack([gx, gy], dim=-1).unsqueeze(0)

    sampled = F.grid_sample(feature.unsqueeze(0), grid, mode="bilinear",

                            align_corners=False)

    return sampled.squeeze(0)

In [ ]:
```

Every number is at a bilinearly-sampled position. No rounding, no quantisation, no dropped gradients.

### Step 2: Compare to torchvision's RoIAlign

In [ ]:
```python

from torchvision.ops import roi_align

feature = torch.randn(1, 16, 50, 50)

boxes = torch.tensor([[0, 10, 20, 100, 90]], dtype=torch.float32)  # (batch_idx, x1, y1, x2, y2)

ours = roi_align_single(feature[0], boxes[0, 1:].tolist(), output_size=7, spatial_scale=1/4)

theirs = roi_align(feature, boxes, output_size=(7, 7), spatial_scale=1/4, sampling_ratio=1, aligned=True)[0]

print(f"shape ours:   {tuple(ours.shape)}")

print(f"shape theirs: {tuple(theirs.shape)}")

print(f"max|diff|:    {(ours - theirs).abs().max().item():.3e}")

In [ ]:
```

With `sampling_ratio=1` and `aligned=True`, the two match to within `1e-5`.

### Step 3: Load a pretrained Mask R-CNN

In [ ]:
```python

import torch

from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights

model = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT)

model.eval()

print(f"params: {sum(p.numel() for p in model.parameters()):,}")

print(f"classes (including background): {len(model.roi_heads.box_predictor.cls_score.out_features * [0])}")

In [ ]:
```

46M parameters, 91 classes (COCO). The first class (id 0) is background; everything the model actually detects starts at id 1.

### Step 4: Run inference

In [ ]:
```python

with torch.no_grad():

    x = torch.randn(3, 400, 600)

    predictions = model([x])

p = predictions[0]

print(f"boxes:  {tuple(p['boxes'].shape)}")

print(f"labels: {tuple(p['labels'].shape)}")

print(f"scores: {tuple(p['scores'].shape)}")

print(f"masks:  {tuple(p['masks'].shape)}")

In [ ]:
```

The mask tensor is shape `(N, 1, H, W)`. Threshold at 0.5 to get a binary mask per object:

In [ ]:
```python

binary_masks = (p['masks'] > 0.5).squeeze(1)  # (N, H, W) boolean

In [ ]:
```

### Step 5: Swap the heads for a custom class count

The common fine-tuning recipe: reuse the backbone, FPN, and RPN; replace the two classifier heads.

In [ ]:
```python

from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def build_custom_maskrcnn(num_classes):

    model = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT)

    in_features = model.roi_heads.box_predictor.cls_score.in_features

    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels

    hidden_layer = 256

    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model

custom = build_custom_maskrcnn(num_classes=5)

print(f"custom cls_score.out_features: {custom.roi_heads.box_predictor.cls_score.out_features}")

In [ ]:
```

`num_classes` must include the background class, so a dataset with 4 object classes uses `num_classes=5`.

### Step 6: Freeze what does not need training

On small datasets, freeze the backbone and the FPN. Only the RPN objectness + regression and the two heads learn.

In [ ]:
```python

def freeze_backbone_and_fpn(model):

    # torchvision Mask R-CNN packs the FPN inside `model.backbone` (as

    # `model.backbone.fpn`), so iterating `model.backbone.parameters()` covers

    # both the ResNet feature layers and the FPN lateral/output convs.

    for p in model.backbone.parameters():

        p.requires_grad = False

    return model

custom = freeze_backbone_and_fpn(custom)

trainable = sum(p.numel() for p in custom.parameters() if p.requires_grad)

print(f"trainable after freeze: {trainable:,}")

In [ ]:
```

On 500-image datasets this is the difference between convergence and overfitting.

## Exercises

In [ ]:
1. **(Easy)** Verify your RoIAlign against `torchvision.ops.roi_align` on 100 random boxes. Report the max absolute difference. Also run RoIPool (pre-2017 behaviour) and show it diverges by ~1-2 feature-map pixels on boxes near the border.
2. **(Medium)** Fine-tune `maskrcnn_resnet50_fpn_v2` on a 50-image custom dataset (any two classes: balloons, fish, pothole, logos). Freeze the backbone, train for 20 epochs, report mask AP@0.5.
3. **(Hard)** Replace Mask R-CNN's mask head with one that predicts at 56x56 instead of 28x28. Measure mAP@IoU=0.75 before and after. Explain why the gain (or lack of one) matches the expected boundary-precision / memory trade-off.